In [1]:
import os
import pandas as pd
import xmltodict
import re
import json

In [2]:
import chardet

# Lê o arquivo em modo binário
with open("./Arquivos/extrato/bb/2024/extrato.ofx", "rb") as f:
    dados = f.read()

# Detecta o encoding
resultado = chardet.detect(dados)

print(resultado)

{'encoding': 'ISO-8859-1', 'confidence': 0.73, 'language': ''}


#### Configurações

In [3]:
path_ofx_cartao = './Arquivos/novo/'
path_ofx_extrato = './Arquivos/extrato/bb/2024'

#### Funções

In [13]:
def get_lista_ofx(path_arquivos):
    contador = 0
    lista_ofx = []
    
    for pasta_atual, subpastas, arquivos in os.walk(path_arquivos):
            for arquivo in arquivos:
                caminho_completo = os.path.join(pasta_atual, arquivo)
                #print(caminho_completo)
                if arquivo.lower().split('.')[-1] == 'ofx':
                    contador += 1
                    lista_ofx.append(os.path.join(pasta_atual, arquivo))
    
    print('Arquivos encontrados: {}'.format(contador))

    return lista_ofx
    
def ofx_to_dict(path_arquivo):
    with open(path_arquivo, "r", encoding='ISO-8859-1') as f:
        conteudo = f.read()
    
    # Extrai apenas o trecho entre <OFX> e </OFX>
    inicio = conteudo.find("<OFX>")
    fim = conteudo.find("</OFX>")
    
    if inicio == -1 or fim == -1:
        raise ValueError("Bloco <OFX>...</OFX> não encontrado no arquivo.")
    
    # Inclui as tags e obtém o conteúdo completo do bloco
    trecho_ofx = conteudo[inicio:fim + len("</OFX>")]
    
    # Converte o conteúdo XML para dicionário
    dados = xmltodict.parse(trecho_ofx)

    return dados
    
def list_nodes(d, prefixo=""):
    chaves = []
    if isinstance(d, dict):
        for k, v in d.items():
            caminho = f"{prefixo}.{k}" if prefixo else k
            chaves.append(caminho)
            chaves.extend(list_nodes(v, caminho))
    elif isinstance(d, list):
        for i, item in enumerate(d):
            caminho = f"{prefixo}[{i}]"
            chaves.extend(list_nodes(item, caminho))
    return chaves

def get_by_node(d, path, default=None, sep="."):
    try:
        for k in path.split(sep):
            if isinstance(d, dict):
                d = d[k]
            elif isinstance(d, list):
                d = d[int(k)]
            else:
                return default
        return d
    except (KeyError, IndexError, ValueError, TypeError):
        return default

def list_nodes_final(dict_ofx):
    nos = [re.sub(r"\[\d+\]", "[X]", p) for p in list_nodes(dict_ofx)]
    lista_nos_finais = []
    
    for i in range(0,len(nos)):
        no = nos[i]    
        l = [p for p in nos if re.match(r"^" + no + ".+", p)]
        
        if len(l) == 0 and no not in lista_nos_finais:
            lista_nos_finais.append(no)    
            print('{}: {}'.format(nos[i],get_by_node(dict_ofx,nos[i])))

    return lista_nos_finais

In [5]:
# get_lista_ofx('./Arquivos/cartão/')
d = ofx_to_dict('./Arquivos/cartão/Elo\\2024\\OUROCARD_ELO_NANQUIM-Fev_24.ofx')

with open("cartao_elo_teste.json", "w", encoding="utf-8") as f:
    json.dump(d, f, ensure_ascii=False)

In [52]:
# Compressão ofx para json 
print('Ganho de ofx para fson:',31.9/40.7)
# participação da lista no arquivo
print('Participação da listagem no arquivo',32258/32751)
# Parquet
print('Parquet sem compressão',10.5/(32258/32751*31.9),10.5/40.7)
print('Parquet com compressão',7.8/10.5,7.8/40.7)
print('Parquet com compressão e campos numero e data',8.22/10.5,7.8/40.7)
# 

Ganho de ofx para fson: 0.7837837837837837
Participação da listagem no arquivo 0.9849470245183353
Parquet sem compressão 0.33418406962205777 0.25798525798525795
Parquet com compressão 0.7428571428571429 0.19164619164619162
Parquet com compressão e campos numero e data 0.7828571428571429 0.19164619164619162


In [105]:
df = pd.DataFrame(get_by_node(d,'OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS.BANKTRANLIST.STMTTRN'))
df['TRNAMT'] = df['TRNAMT'].astype(float)
df["DTPOSTED"] = pd.to_datetime(df["DTPOSTED"].astype(str), format="%Y%m%d")

In [106]:
# 
df['parc'] = df[(df['MEMO'].str.contains('PARC')) & (df['TRNAMT'] < 0) & ~(df['MEMO'].str.contains('ANUIDADE '))]['MEMO'].str[19:19+2].astype(int)
df['nparc'] = df[(df['MEMO'].str.contains('PARC')) & (df['TRNAMT'] < 0) & ~(df['MEMO'].str.contains('ANUIDADE '))]['MEMO'].str[19+3:19+2+3].astype(int)

In [117]:
df['parc_restante'] = (df['nparc'] - df['parc'])

In [122]:
df.groupby('parc_restante')['TRNAMT'].sum()

parc_restante
0.0   -2666.60
1.0   -3351.35
2.0   -1214.03
3.0    -289.93
4.0    -859.11
5.0    -427.83
6.0    -214.67
8.0    -566.90
Name: TRNAMT, dtype: float64

In [49]:
df.memory_usage(deep=True)

Index         132
TRNTYPE     10858
DTPOSTED     1552
TRNAMT       1552
FITID       16102
MEMO        17073
dtype: int64

In [38]:
df.memory_usage(deep=True)

Index         132
TRNTYPE     10858
DTPOSTED    11058
TRNAMT      10728
FITID       16102
MEMO        17073
dtype: int64

In [51]:
df.to_parquet('cartao_elo_teste.parquet',compression='brotli')

# Análise

In [60]:
lista_ofx_cartao = get_lista_ofx(path_ofx_cartao)
lista_ofx_extrato = get_lista_ofx(path_ofx_extrato)

Arquivos encontrados: 6
Arquivos encontrados: 14


In [61]:
dict_cartao = ofx_to_dict(lista_ofx_cartao[0])
dict_extrato = ofx_to_dict(lista_ofx_extrato[0])

In [62]:
_ = list_nodes_final(dict_cartao)

OFX.SIGNONMSGSRSV1.SONRS.STATUS.CODE: 0
OFX.SIGNONMSGSRSV1.SONRS.STATUS.SEVERITY: INFO
OFX.SIGNONMSGSRSV1.SONRS.DTSERVER: 20251013211110
OFX.SIGNONMSGSRSV1.SONRS.LANGUAGE: POR
OFX.SIGNONMSGSRSV1.SONRS.FI.ORG: Banco do Brasil
OFX.SIGNONMSGSRSV1.SONRS.FI.FID: 1
OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.TRNUID: 1
OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.STATUS.CODE: 0
OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.STATUS.SEVERITY: INFO
OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS.CURDEF: BRL
OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS.CCACCTFROM.ACCTID: 6516000000004407
OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS.BANKTRANLIST.DTSTART: 20241007
OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS.BANKTRANLIST.DTEND: 20250929
OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS.BANKTRANLIST.STMTTRN[X].TRNTYPE: None
OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS.BANKTRANLIST.STMTTRN[X].DTPOSTED: None
OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS.BANKTRANLIST.STMTTRN[X].TRNAMT: None
OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS.BANKTRANLIST.S

In [37]:
# get_by_node(dict_cartao,'OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS.BANKTRANLIST.STMTTRN')

In [34]:
# ofx comum
{
# Signon Message Set Response, version 1", ou seja, é o bloco de resposta de autenticação (ou "sessão de login")
'ss_code':     {'path':['OFX.SIGNONMSGSRSV1.SONRS.STATUS.CODE'],'type':'str'}, #: 0
'ss_severity': {'path':['OFX.SIGNONMSGSRSV1.SONRS.STATUS.SEVERITY'],'type':'str'}, #: INFO
'ss_dtserver': {'path':['OFX.SIGNONMSGSRSV1.SONRS.DTSERVER'],'type':'str'}, #: 20251013211110 20240509120000[-3:BRT]
'ss_language': {'path':['OFX.SIGNONMSGSRSV1.SONRS.LANGUAGE'],'type':'str'}, #: POR
'ss_fi_org':   {'path':['OFX.SIGNONMSGSRSV1.SONRS.FI.ORG'],'type':'str'}, #: Banco do Brasil
'ss_fi_fid':   {'path':['OFX.SIGNONMSGSRSV1.SONRS.FI.FID'],'type':'str'}, #: 1

# CREDITCARDMSGSRSV1 Credit Card Message Set Response, version 1
# CCSTMTTRNRS Credit Card Statement Transaction Response
# BANKMSGSRSV1 Bank Message Set Response, version 1
# STMTTRNRS Statement Transaction Response
'ss_rsp_trunid':    {'path':['OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.TRNUID',
                              'OFX.BANKMSGSRSV1.STMTTRNRS.TRNUID'],'type':'str'}, #: 1
'ss_rsp_code':      {'path':['OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.STATUS.CODE',
                              'OFX.BANKMSGSRSV1.STMTTRNRS.STATUS.CODE'],'type':'str'}, #: 0
'ss_rsp_serverity': {'path':['OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.STATUS.SEVERITY',
                              'OFX.BANKMSGSRSV1.STMTTRNRS.STATUS.SEVERITY'],'type':'str'}, #: INFO
}

{'sessao_code': {'path': ['OFX.SIGNONMSGSRSV1.SONRS.STATUS.CODE'],
  'type': 'str'},
 'sessao_severity': {'path': ['OFX.SIGNONMSGSRSV1.SONRS.STATUS.SEVERITY'],
  'type': 'str'},
 'sessao_dtserver': {'path': ['OFX.SIGNONMSGSRSV1.SONRS.DTSERVER'],
  'type': 'str'},
 'sessao_language': {'path': ['OFX.SIGNONMSGSRSV1.SONRS.LANGUAGE'],
  'type': 'str'},
 'sessao_fi_org': {'path': ['OFX.SIGNONMSGSRSV1.SONRS.FI.ORG'], 'type': 'str'},
 'sessao_fi_fid': {'path': ['OFX.SIGNONMSGSRSV1.SONRS.FI.FID'], 'type': 'str'},
 'setresp_trunid': {'path': ['OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.TRNUID',
   'OFX.BANKMSGSRSV1.STMTTRNRS.TRNUID'],
  'type': 'str'},
 'setresp_code': {'path': ['OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.STATUS.CODE',
   'OFX.BANKMSGSRSV1.STMTTRNRS.STATUS.CODE'],
  'type': 'str'},
 'setresp_serverity': {'path': ['OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.STATUS.SEVERITY',
   'OFX.BANKMSGSRSV1.STMTTRNRS.STATUS.SEVERITY'],
  'type': 'str'}}

In [64]:
# CCSTMTRS Credit Card Statement Response
{
'cc_curdef':  {'path':['OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS.CURDEF'],'type':'str'}, #: BRL
'cc_acctid':  {'path':['OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS.CCACCTFROM.ACCTID'],'type':'str'}, #: 6516000000004407
'cc_dtstart': {'path':['OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS.BANKTRANLIST.DTSTART'],'type':'str'}, #: 20241007
'cc_dtend':   {'path':['OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS.BANKTRANLIST.DTEND'],'type':'str'}, #: 20250929
'cc_balamt':  {'path':['OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS.LEDGERBAL.BALAMT'],'type':'str'}, #: -23361.21
'cc_dtasof':  {'path':['OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS.LEDGERBAL.DTASOF'],'type':'str'}, #: 20251010
}

# Listagem
# Base OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS.BANKTRANLIST.STMTTRN
# STMTTRN Statement Transaction — ou seja, uma transação individua
{
'cc_tr_trntype':  {'key':['TRNTYPE'],'type':'str'}, #: None
'cc_tr_dtposted':  {'key':['DTPOSTED'],'type':'str'}, #: None
'cc_tr_trnamt':  {'key':['TRNAMT'],'type':'str'}, #: None
'cc_tr_fitid':  {'key':['FITID'],'type':'str'}, #: None
'cc_tr_memo':  {'key':['MEMO'],'type':'str'}, #: None
}

{'cc_tr_trntype': {'key': ['TRNTYPE'], 'type': 'str'},
 'cc_tr_dtposted': {'key': ['DTPOSTED'], 'type': 'str'},
 'cc_tr_trnamt': {'key': ['TRNAMT'], 'type': 'str'},
 'cc_tr_fitid': {'key': ['FITID'], 'type': 'str'},
 'cc_tr_memo': {'key': ['MEMO'], 'type': 'str'}}

In [ ]:
#### parei aqui

In [ ]:
# STMTRS Statement Response (Resposta de Extrato Bancário)
{
'bnk_curdef':  {'path':['OFX.BANKMSGSRSV1.STMTTRNRS.STMTRS.CURDEF'],'type':'str'}, #: BRL
'bnk_bankid':  {'path':['OFX.BANKMSGSRSV1.STMTTRNRS.STMTRS.BANKACCTFROM.BANKID'],'type':'str'}, #: 1
'bnk_curdef':  {'path':['OFX.BANKMSGSRSV1.STMTTRNRS.STMTRS.BANKACCTFROM.BRANCHID'],'type':'str'}, #: 5929-3
'bnk_curdef':  {'path':['OFX.BANKMSGSRSV1.STMTTRNRS.STMTRS.BANKACCTFROM.ACCTID'],'type':'str'}, #: 94944-2
'bnk_curdef':  {'path':['OFX.BANKMSGSRSV1.STMTTRNRS.STMTRS.BANKACCTFROM.ACCTTYPE'],'type':'str'}, #: CHECKING
'bnk_curdef':  {'path':['OFX.BANKMSGSRSV1.STMTTRNRS.STMTRS.BANKTRANLIST.DTSTART'],'type':'str'}, #: 20240131120000[-3:BRT]
'bnk_curdef':  {'path':['OFX.BANKMSGSRSV1.STMTTRNRS.STMTRS.BANKTRANLIST.DTEND'],'type':'str'}, #: 20240229120000[-3:BRT]
'bnk_curdef':  {'path':['OFX.BANKMSGSRSV1.STMTTRNRS.STMTRS.LEDGERBAL.BALAMT'],'type':'str'}, #: 18642.66
'bnk_curdef':  {'path':['OFX.BANKMSGSRSV1.STMTTRNRS.STMTRS.LEDGERBAL.DTASOF'],'type':'str'}, #: 20240229120000[-3:BRT]
}

# OFX.BANKMSGSRSV1.STMTTRNRS.STMTRS.BANKTRANLIST.STMTTRN
'cartao_registro_trntype':  {'key':['TRNTYPE'],'type':'str'}, #: None
'cartao_registro_trntype':  {'key':['DTPOSTED'],'type':'str'}, #: None
'cartao_registro_trntype':  {'key':['TRNAMT'],'type':'str'}, #: None
'cartao_registro_trntype':  {'key':['FITID'],'type':'str'}, #: None
'cartao_registro_trntype':  {'key':['CHECKNUM'],'type':'str'}, #: None
'cartao_registro_trntype':  {'key':['REFNUM'],'type':'str'}, #: None
'cartao_registro_trntype':  {'key':['MEMO'],'type':'str'}, #: None

In [63]:
_ = list_nodes_final(dict_extrato)

OFX.SIGNONMSGSRSV1.SONRS.STATUS.CODE: 0
OFX.SIGNONMSGSRSV1.SONRS.STATUS.SEVERITY: INFO
OFX.SIGNONMSGSRSV1.SONRS.DTSERVER: 20240509120000[-3:BRT]
OFX.SIGNONMSGSRSV1.SONRS.LANGUAGE: POR
OFX.SIGNONMSGSRSV1.SONRS.FI.ORG: Banco do Brasil
OFX.SIGNONMSGSRSV1.SONRS.FI.FID: 1
OFX.BANKMSGSRSV1.STMTTRNRS.TRNUID: 1
OFX.BANKMSGSRSV1.STMTTRNRS.STATUS.CODE: 0
OFX.BANKMSGSRSV1.STMTTRNRS.STATUS.SEVERITY: INFO
OFX.BANKMSGSRSV1.STMTTRNRS.STMTRS.CURDEF: BRL
OFX.BANKMSGSRSV1.STMTTRNRS.STMTRS.BANKACCTFROM.BANKID: 1
OFX.BANKMSGSRSV1.STMTTRNRS.STMTRS.BANKACCTFROM.BRANCHID: 5929-3
OFX.BANKMSGSRSV1.STMTTRNRS.STMTRS.BANKACCTFROM.ACCTID: 94944-2
OFX.BANKMSGSRSV1.STMTTRNRS.STMTRS.BANKACCTFROM.ACCTTYPE: CHECKING
OFX.BANKMSGSRSV1.STMTTRNRS.STMTRS.BANKTRANLIST.DTSTART: 20240131120000[-3:BRT]
OFX.BANKMSGSRSV1.STMTTRNRS.STMTRS.BANKTRANLIST.DTEND: 20240229120000[-3:BRT]
OFX.BANKMSGSRSV1.STMTTRNRS.STMTRS.BANKTRANLIST.STMTTRN[X].TRNTYPE: None
OFX.BANKMSGSRSV1.STMTTRNRS.STMTRS.BANKTRANLIST.STMTTRN[X].DTPOSTED: None
OFX.BAN

In [75]:
# ['OFX']
#   ['SIGNONMSGSRSV1']
#     ['SONRS']  
#       ['STATUS']  
#         ['CODE']: 0
#         ['SEVERITY']: INFO
#       ['DTSERVER']: 20251013211110
#       ['LANGUAGE']: POR
#       ['FI']
#         ['ORG']: 'Banco do Brasil'
#         ['FID']: '1'
#   ['CREDITCARDMSGSRSV1']
#     ['CCSTMTTRNRS']
#       ['TRNUID']: '1'
#       ['STATUS']
#         ['CODE']: '0'
#         ['SEVERITY']: 'INFO'
#       ['CCSTMTRS']
#         ['CURDEF']: 'BRL'
#         ['CCACCTFROM']
#           ['ACCTID']: '6516000000004407'
#         ['BANKTRANLIST']
#           ['DTSTART']: '20241007'
#           ['DTEND',]: '20250929'
#           ['STMTTRN'] >>> Lista
#         ['LEDGERBAL']
#           ['BALAMT']: '-23361.21'
#           ['DTASOF']: '20251010'
header = {
    'CODE': dados['OFX']['SIGNONMSGSRSV1']['SONRS']['STATUS']['CODE'],
    'SEVERITY': dados['OFX']['SIGNONMSGSRSV1']['SONRS']['STATUS']['SEVERITY'],
    'DTSERVER': dados['OFX']['SIGNONMSGSRSV1']['SONRS']['DTSERVER'],
    'LANGUAGE': dados['OFX']['SIGNONMSGSRSV1']['SONRS']['LANGUAGE'],
    'ORG': dados['OFX']['SIGNONMSGSRSV1']['SONRS']['FI']['ORG'],
    'FID': dados['OFX']['SIGNONMSGSRSV1']['SONRS']['FI']['FID'],
}
# print(dados['OFX']['CREDITCARDMSGSRSV1']['CCSTMTTRNRS'])
# print(dados['OFX']['CREDITCARDMSGSRSV1']['CCSTMTTRNRS']['CCSTMTRS']['BANKTRANLIST']['STMTTRN'])

In [79]:
get_by_path(dados,'OFX.SIGNONMSGSRSV1.SONRS.STATUS.SEVERITY')

'INFO'

In [77]:
listar_chaves(dados)

['OFX',
 'OFX.SIGNONMSGSRSV1',
 'OFX.SIGNONMSGSRSV1.SONRS',
 'OFX.SIGNONMSGSRSV1.SONRS.STATUS',
 'OFX.SIGNONMSGSRSV1.SONRS.STATUS.CODE',
 'OFX.SIGNONMSGSRSV1.SONRS.STATUS.SEVERITY',
 'OFX.SIGNONMSGSRSV1.SONRS.DTSERVER',
 'OFX.SIGNONMSGSRSV1.SONRS.LANGUAGE',
 'OFX.SIGNONMSGSRSV1.SONRS.FI',
 'OFX.SIGNONMSGSRSV1.SONRS.FI.ORG',
 'OFX.SIGNONMSGSRSV1.SONRS.FI.FID',
 'OFX.CREDITCARDMSGSRSV1',
 'OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS',
 'OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.TRNUID',
 'OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.STATUS',
 'OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.STATUS.CODE',
 'OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.STATUS.SEVERITY',
 'OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS',
 'OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS.CURDEF',
 'OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS.CCACCTFROM',
 'OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS.CCACCTFROM.ACCTID',
 'OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS.BANKTRANLIST',
 'OFX.CREDITCARDMSGSRSV1.CCSTMTTRNRS.CCSTMTRS.BANKTRANLIST.DTSTART',
 'OFX.CREDITCARD

In [ ]:
print(d)
for t in dados['OFX']['CREDITCARDMSGSRSV1']:
    print(t)

dados['OFX']['CREDITCARDMSGSRSV1']

{'CODE': '0', 'SEVERITY': 'INFO', 'DTSERVER': '20251013211110', 'LANGUAGE': 'POR', 'ORG': 'Banco do Brasil', 'FID': '1'}
CCSTMTTRNRS


{'CCSTMTTRNRS': {'TRNUID': '1',
  'STATUS': {'CODE': '0', 'SEVERITY': 'INFO'},
  'CCSTMTRS': {'CURDEF': 'BRL',
   'CCACCTFROM': {'ACCTID': '6516000000004407'},
   'BANKTRANLIST': {'DTSTART': '20241007',
    'DTEND': '20250929',
    'STMTTRN': [{'TRNTYPE': 'CREDIT',
      'DTPOSTED': '20250929',
      'TRNAMT': '8.80',
      'FITID': '2025092965160000000044070000000000',
      'MEMO': 'DESC AUTOMATICO ANUD. TIT-PARC 07/12 BR'},
     {'TRNTYPE': 'CREDIT',
      'DTPOSTED': '20250910',
      'TRNAMT': '20904.91',
      'FITID': '2025091065160000000044070000000001',
      'MEMO': 'PGTO DEBITO CONTA 5929 000004732  200211'},
     {'TRNTYPE': 'PAYMENT',
      'DTPOSTED': '20250923',
      'TRNAMT': '-327.00',
      'FITID': '2025092365160000000044070000000002',
      'MEMO': 'PG CONCURSOS INTELIGE   554799911364 BR'},
     {'TRNTYPE': 'PAYMENT',
      'DTPOSTED': '20250829',
      'TRNAMT': '-34.80',
      'FITID': '2025082965160000000044070000000003',
      'MEMO': 'BRASILIA GUARA         B

In [ ]:
# Agora 'dados' é um dict Python
print(type(dados))
print(dados)

<class 'dict'>
{'OFX': {'SIGNONMSGSRSV1': {'SONRS': {'STATUS': {'CODE': '0', 'SEVERITY': 'INFO'}, 'DTSERVER': '20251013211110', 'LANGUAGE': 'POR', 'FI': {'ORG': 'Banco do Brasil', 'FID': '1'}}}, 'CREDITCARDMSGSRSV1': {'CCSTMTTRNRS': {'TRNUID': '1', 'STATUS': {'CODE': '0', 'SEVERITY': 'INFO'}, 'CCSTMTRS': {'CURDEF': 'BRL', 'CCACCTFROM': {'ACCTID': '6516000000004407'}, 'BANKTRANLIST': {'DTSTART': '20241007', 'DTEND': '20250929', 'STMTTRN': [{'TRNTYPE': 'CREDIT', 'DTPOSTED': '20250929', 'TRNAMT': '8.80', 'FITID': '2025092965160000000044070000000000', 'MEMO': 'DESC AUTOMATICO ANUD. TIT-PARC 07/12 BR'}, {'TRNTYPE': 'CREDIT', 'DTPOSTED': '20250910', 'TRNAMT': '20904.91', 'FITID': '2025091065160000000044070000000001', 'MEMO': 'PGTO DEBITO CONTA 5929 000004732  200211'}, {'TRNTYPE': 'PAYMENT', 'DTPOSTED': '20250923', 'TRNAMT': '-327.00', 'FITID': '2025092365160000000044070000000002', 'MEMO': 'PG CONCURSOS INTELIGE   554799911364 BR'}, {'TRNTYPE': 'PAYMENT', 'DTPOSTED': '20250829', 'TRNAMT': '-